<a href="https://colab.research.google.com/github/Sanjaysanjay31/24B11DS095_DriveReady_OPC_2026/blob/main/Day_2/Startup_Marketing_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 2 — Prompts, Memory, and Python Fundamentals for Agents

**Project:** AI Startup Marketing Agent  
**Curriculum:** Aditya University — One Person Company (12-Day Agentic AI Workshop)  
**Primary Provider:** OpenRouter — NVIDIA Nemotron 3.5 Super VL (Free)  
**Fallback Provider:** Groq — GPT-OSS 120B  

This notebook implements Day 2 fundamentals: structured system prompts, few-shot classification, JSON data extraction, sliding-window conversation memory, and essential Python primitives (dictionaries, type hints, decorators, try/except). Supports both Google Colab Secrets and local `.env` execution.

In [1]:
# Install OpenAI client and python-dotenv for dual Colab and local execution
!pip install -q openai python-dotenv

In [2]:
import os
import json
from openai import OpenAI

# Load environment variables (.env locally or Colab Secrets)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

try:
    from google.colab import userdata
    colab_openrouter = userdata.get("OPENROUTER_API_KEY")
    colab_groq = userdata.get("GROQ_API_KEY") or userdata.get("agentic_ai_GROQ_API_KEY")
except Exception:
    colab_openrouter = None
    colab_groq = None

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY") or colab_openrouter
GROQ_API_KEY = os.getenv("GROQ_API_KEY") or colab_groq

openrouter_client = OpenAI(api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1") if OPENROUTER_API_KEY else None
groq_client = OpenAI(api_key=GROQ_API_KEY, base_url="https://api.groq.com/openai/v1") if GROQ_API_KEY else None

PRIMARY_MODEL = "nvidia/nemotron-3.5-super-vl:free"
FALLBACK_MODEL = "openai/gpt-oss-120b"

print("API setup complete: OpenRouter (Primary) + Groq (Fallback)")

API setup complete: OpenRouter (Primary) + Groq (Fallback)


In [3]:
# Marketing service catalogue with pricing in Indian Rupees (INR)
MARKETING_SERVICES = {
    "viral_hook_generation": 400,
    "short_script_writing": 700,
    "storyboard_generation": 1200,
    "voiceover_synthesis": 500,
    "auto_subtitle_karaoke": 300,
    "vertical_video_assembly": 1500,
    "seo_metadata_pack": 350
}

SYSTEM = """You are the AI Startup Marketing Agent for early-stage founders.
Services & Prices: Hook Rs 400, Script Rs 700, Storyboard Rs 1200, Voiceover Rs 500, Subtitles Rs 300, Video Assembly Rs 1500, SEO Pack Rs 350.
Rules: Never guarantee viral views or follower numbers. Quote prices strictly from the catalogue. If unknown, say 'I will check with the team.'
Format: Under 50 words, practical, encouraging."""

def ask_llm(messages):
    """Queries primary OpenRouter model with automatic fallback to Groq."""
    if openrouter_client:
        try:
            res = openrouter_client.chat.completions.create(model=PRIMARY_MODEL, messages=messages, temperature=0.3)
            return res.choices[0].message.content
        except Exception as e:
            print(f"[Failover] Switching to Groq: {type(e).__name__}")
    if groq_client:
        res = groq_client.chat.completions.create(model=FALLBACK_MODEL, messages=messages, temperature=0.3)
        return res.choices[0].message.content
    raise RuntimeError("Configure OPENROUTER_API_KEY or GROQ_API_KEY.")

def chat(user_message):
    return ask_llm([{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_message}])

print(chat("What services do you provide for short-form video marketing?"))

We provide short-form video marketing services: Viral Hook Generation (Rs 400), Short Script Writing (Rs 700), Storyboard Generation (Rs 1200), Voiceover Synthesis (Rs 500), Auto Subtitle Karaoke (Rs 300), Vertical Video Assembly (Rs 1500), and SEO Metadata Packs (Rs 350). All tailored for faceless 9:16 Shorts and Reels!


In [4]:
# 1. Few-shot request classifier
CLASSIFIER_PROMPT = """Classify into one category: hook_creation, scriptwriting, storyboard, voiceover, video_assembly, pricing_inquiry, other.
Examples:
Customer: Opening line for my dev tool -> hook_creation
Customer: 30-second script about expense reports -> scriptwriting
Customer: How much is video assembly? -> pricing_inquiry
Return only the category name."""

def classify_request(message):
    return ask_llm([{"role": "system", "content": CLASSIFIER_PROMPT}, {"role": "user", "content": message}]).strip().lower()

# 2. Structured JSON brief extractor
def extract_brief(message):
    prompt = """Extract into JSON: {"founder_name": "", "startup_name": "", "services_requested": [], "target_problem": ""}
    Rules: Valid JSON only. Do not invent missing details."""
    res = ask_llm([{"role": "system", "content": prompt}, {"role": "user", "content": message}]).strip()
    if res.startswith("```"):
        res = res.replace("```json", "", 1).replace("```", "").strip()
    return json.loads(res)

print("Classification:", classify_request("I need a viral hook to stop founders scrolling."))
sample_brief = "I'm Sanjay from DevPulse. Need viral hook generation. Developers ignore code reviews."
print("Extracted Brief:\n", json.dumps(extract_brief(sample_brief), indent=2))

Classification: hook_creation
Extracted Brief:
{
  "founder_name": "Sanjay",
  "startup_name": "DevPulse",
  "services_requested": [
    "viral_hook_generation"
  ],
  "target_problem": "Developers ignore code reviews"
}


In [5]:
import functools

def log_call(func):
    """Decorator logging function calls and results."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"   [log] {func.__name__}({list(args[0]) if args else ''}) -> Rs {result}")
        return result
    return wrapper

@log_call
def calculate_cost(services: list[str]) -> int:
    """Calculate total campaign package cost for requested services."""
    total = 0
    for s in services:
        if s not in MARKETING_SERVICES:
            raise ValueError(f"Unknown service: {s}")
        total += MARKETING_SERVICES[s]
    return total

selected = ["viral_hook_generation", "short_script_writing", "vertical_video_assembly"]
print("Total Campaign Quote: Rs", calculate_cost(selected))

   [log] calculate_cost(['viral_hook_generation', 'short_script_writing', 'vertical_video_assembly']) -> Rs 2600
Total Campaign Quote: Rs 2600


In [6]:
from openai import RateLimitError, AuthenticationError, APIConnectionError

history = []
MAX_TURNS = 5

def chat_with_memory(user_message: str) -> str:
    """Maintains rolling conversational memory with rollback on error."""
    history.append({"role": "user", "content": user_message})
    recent = history[-(MAX_TURNS * 2):]
    messages = [{"role": "system", "content": SYSTEM}] + recent

    try:
        reply = ask_llm(messages)
        history.append({"role": "assistant", "content": reply})
        return reply
    except RateLimitError:
        return "Server is busy. Please try again in a minute."
    except APIConnectionError:
        return "Connection issue. Please verify internet connectivity."
    except Exception as e:
        history.pop()
        return f"Error: {e}"

# Demonstrate memory retention
print(chat_with_memory("Hi, I'm Sanjay, founder of DevPulse."))
print(chat_with_memory("We build an automated AI code review assistant."))
print(chat_with_memory("What is my name and startup?"))

Hello Sanjay! Welcome. DevPulse sounds like an impactful product. How can I help you create viral short-form videos today?
Automated code review addresses an acute pain point—developer fatigue and bug leaks. We can frame hooks highlighting production crash stress.
Your name is Sanjay, and you run DevPulse, an automated AI code review assistant!


In [7]:
# Automated multi-turn dialogue verification
test_prompts = [
    "What services help create a 30-second Reel?",
    "How much for short script writing and video assembly?",
]

for p in test_prompts:
    print("\nUser:", p)
    print("Agent:", chat_with_memory(p))



User: What services help create a 30-second Reel?
Agent: I recommend Viral Hook Generation (Rs 400), Short Script Writing (Rs 700), and Vertical Video Assembly (Rs 1500) for a ready-to-post Reel!

User: How much for short script writing and video assembly?
Agent: Short Script Writing is Rs 700 and Vertical Video Assembly is Rs 1500, making the total Rs 2200.
